<a href="https://colab.research.google.com/github/1900690/cucumber/blob/main/fastlabel-ultraliticshub-boundingbox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#zipを解凍
import shutil
import os

file_name ="cucanber-fruit-flower_20260409143838.zip"

#データを解凍
shutil.unpack_archive('/content/'+file_name, '/content/')
#zipを消す
os.remove('/content/'+file_name)

In [ ]:
# @title  データセット構成とオーグメンテーション設定
import os
import yaml
import shutil
import cv2
import numpy as np
import albumentations as A
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

# --- パス設定 ---
# @markdown ###  パスと分割の設定
source_img_dir = '/content/original' # @param {type:"string"}
source_label_dir = '/content/yolo/annotations' # @param {type:"string"}
classes_file = '/content/yolo/classes.txt' # @param {type:"string"}
output_root_dir = '/content/dataset/detect' # @param {type:"string"}

# @markdown ---
# @markdown ###  分割比率の指定
train_ratio = 0.7 # @param {type:"slider", min:0, max:1, step:0.1}
val_ratio = 0.2 # @param {type:"slider", min:0, max:1, step:0.1}
test_ratio = 0.1 # @param {type:"slider", min:0, max:1, step:0.1}

# --- オーグメンテーション詳細設定 ---
# @markdown ---
# @markdown ###  オーグメンテーション適用設定
apply_to_train = True # @param {type:"boolean"}
apply_to_val = False # @param {type:"boolean"}
apply_to_test = False # @param {type:"boolean"}
# @markdown **1枚の画像から追加生成する枚数**
num_augmented_per_image = 2 # @param {type:"integer"}

# @markdown ---
# @markdown ###  加工メニューの選択
# @markdown - **幾何学的変化 (Bboxも追従します)**
use_rotation = False # @param {type:"boolean"}
use_translation = False # @param {type:"boolean"}
# @markdown - **画質・色の変化**
use_color_jitter = True # @param {type:"boolean"}
use_noise = True # @param {type:"boolean"}
use_blur = True # @param {type:"boolean"}

def get_augmentation_pipeline():
    transforms = []
    if use_rotation: transforms.append(A.Rotate(limit=30, p=0.5))
    if use_translation: transforms.append(A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=0, p=0.5))
    if use_color_jitter: transforms.append(A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5))
    if use_noise: transforms.append(A.GaussNoise(var_limit=(10.0, 50.0), p=0.3))
    if use_blur: transforms.append(A.Blur(blur_limit=3, p=0.3))

    return A.Compose(transforms, bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

def load_yolo_labels(path):
    bboxes, class_labels = [], []
    if os.path.exists(path):
        with open(path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) < 5: continue
                class_labels.append(int(parts[0]))
                bboxes.append(list(map(float, parts[1:])))
    return bboxes, class_labels

def save_yolo_labels(path, bboxes, class_labels):
    with open(path, 'w') as f:
        for label, bbox in zip(class_labels, bboxes):
            f.write(f"{label} {' '.join([f'{x:.6f}' for x in bbox])}\n")

def process_dataset():
    # 安全策：比率の合計チェック
    if abs((train_ratio + val_ratio + test_ratio) - 1.0) > 1e-5:
        print(" 警告: 比率の合計を1.0にしてください。")
        return

    # フォルダリセット
    shutil.rmtree(output_root_dir, ignore_errors=True)

    # クラス名読み込み
    if not os.path.exists(classes_file):
        print(f" エラー: {classes_file} が見つかりません。")
        return
    with open(classes_file, 'r') as f:
        class_names = [line.strip() for line in f.readlines() if line.strip()]

    # 画像取得
    img_exts = ('.jpg', '.jpeg', '.png', '.JPG', '.PNG')
    all_images = [f for f in os.listdir(source_img_dir) if f.endswith(img_exts)]

    # データ分割
    train_files, temp_files = train_test_split(all_images, test_size=(1 - train_ratio), random_state=42)
    rel_val_ratio = val_ratio / (val_ratio + test_ratio) if (val_ratio + test_ratio) > 0 else 0
    val_files, test_files = train_test_split(temp_files, test_size=(1 - rel_val_ratio), random_state=42) if len(temp_files) > 1 else (temp_files, [])

    split_dict = {'train': train_files, 'val': val_files, 'test': test_files}
    aug_settings = {'train': apply_to_train, 'val': apply_to_val, 'test': apply_to_test}
    aug_pipeline = get_augmentation_pipeline()

    # 構造作成
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(output_root_dir, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(output_root_dir, 'labels', split), exist_ok=True)

    print(f"---  処理開始 (Class数: {len(class_names)}) ---")

    for split, files in split_dict.items():
        if not files: continue
        for filename in tqdm(files, desc=f"Split: {split}"):
            basename = os.path.splitext(filename)[0]
            img_src = os.path.join(source_img_dir, filename)
            lbl_src = os.path.join(source_label_dir, basename + '.txt')

            image = cv2.imread(img_src)
            if image is None: continue
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            bboxes, class_labels = load_yolo_labels(lbl_src)

            # --- オリジナル保存 ---
            cv2.imwrite(os.path.join(output_root_dir, 'images', split, filename), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
            if os.path.exists(lbl_src):
                shutil.copy2(lbl_src, os.path.join(output_root_dir, 'labels', split, basename + '.txt'))

            # --- オーグメンテーション実行 ---
            if aug_settings[split]:
                for i in range(num_augmented_per_image):
                    try:
                        augmented = aug_pipeline(image=image, bboxes=bboxes, class_labels=class_labels)
                        if len(augmented['bboxes']) > 0:
                            aug_name = f"{basename}_aug_{i}"
                            # 画像書き出し
                            cv2.imwrite(os.path.join(output_root_dir, 'images', split, f"{aug_name}.jpg"),
                                        cv2.cvtColor(augmented['image'], cv2.COLOR_RGB2BGR))
                            # ラベル書き出し
                            save_yolo_labels(os.path.join(output_root_dir, 'labels', split, f"{aug_name}.txt"),
                                             augmented['bboxes'], class_labels)
                    except Exception as e:
                        continue

    # --- YAML作成 ---
    yaml_data = {
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'names': {i: name for i, name in enumerate(class_names)}
    }
    with open(os.path.join(output_root_dir, 'detect.yaml'), 'w') as f:
        yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

    print(f"\n すべての処理が完了しました！")
    print(f"フォルダ: {output_root_dir}")

# 実行
process_dataset()

In [ ]:
# @title  ZIP圧縮してダウンロード
import shutil
import os
from google.colab import files

# 圧縮したいフォルダのパス（前のセルで指定した output_root_dir）
target_dir = '/content/dataset/detect' # @param {type:"string"}
# 作成するZIPファイル名（拡張子なし）
zip_filename = 'yolo_detect_dataset' # @param {type:"string"}

if os.path.exists(target_dir):
    print(f" {target_dir} を圧縮しています...")

    # フォルダをZIP圧縮
    # shutil.make_archive(出力名, 形式, 圧縮したいディレクトリ)
    shutil.make_archive(zip_filename, 'zip', target_dir)

    zip_path = f"{zip_filename}.zip"
    print(f" 圧縮完了: {zip_path}")

    # ダウンロード開始
    print(" ダウンロードを開始します...")
    files.download(zip_path)
else:
    print(f" エラー: {target_dir} が見つかりません。前のセルを先に実行してください。")

In [ ]:
# @title 📊 データセット分割・拡張・YAML生成
import os
import yaml
import shutil
import cv2
import numpy as np
import albumentations as A
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import math

# --- パス設定 ---
# @markdown ### 📂 入力パスの設定
originals = '/content/original' # @param {type:"string"}
annotations = '/content/yolo/annotations' # @param {type:"string"}
classes_file = '/content/yolo/classes.txt' # @param {type:"string"}
output_root = '/content/datasets' # @param {type:"string"}

# --- 分割比率 ---
# @markdown ### 📈 分割割合の設定
# @markdown ※初めに全体からvalidを切り出し、残りのtrain_testからtestを切り出します
valid分割割合 = 0.2 # @param {type:"slider", min:0, max:1, step:0.01}
test分割割合 = 0.1 # @param {type:"slider", min:0, max:1, step:0.01}

# --- オーグメンテーション設定 ---
# @markdown ### 🎨 拡張設定
apply_augmentation = True # @param {type:"boolean"}
# @markdown **訓練データ1枚から追加生成する枚数**
num_augmented_per_image = 2 # @param {type:"integer"}

# @markdown ---
# @markdown ### 🛠️ 加工メニュー (Bbox追従)
use_rotation = True # @param {type:"boolean"}
use_translation = True # @param {type:"boolean"}
use_color_jitter = True # @param {type:"boolean"}
use_noise = True # @param {type:"boolean"}

def get_aug_pipeline():
    transforms = []
    if use_rotation: transforms.append(A.Rotate(limit=30, p=0.5))
    if use_translation: transforms.append(A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=0, p=0.5))
    if use_color_jitter: transforms.append(A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5))
    if use_noise: transforms.append(A.GaussNoise(var_limit=(10.0, 50.0), p=0.3))
    return A.Compose(transforms, bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))

def load_yolo_labels(path):
    bboxes, class_labels = [], []
    if os.path.exists(path):
        with open(path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) < 5: continue
                class_labels.append(int(parts[0]))
                bboxes.append(list(map(float, parts[1:])))
    return bboxes, class_labels

def save_yolo_labels(path, bboxes, class_labels):
    with open(path, 'w') as f:
        for label, bbox in zip(class_labels, bboxes):
            f.write(f"{label} {' '.join([f'{x:.6f}' for x in bbox])}\n")

# 1. フォルダの初期化 (コード1準拠)
if os.path.exists(output_root):
    shutil.rmtree(output_root)

paths = {
    'train': {'images': f'{output_root}/train/images', 'labels': f'{output_root}/train/labels'},
    'valid': {'images': f'{output_root}/valid/images', 'labels': f'{output_root}/valid/labels'},
    'test':  {'images': f'{output_root}/test/images',  'labels': f'{output_root}/test/labels'}
}

for split in paths.values():
    os.makedirs(split['images'], exist_ok=True)
    os.makedirs(split['labels'], exist_ok=True)

# 2. ファイルリストの準備
read_files_annotations = sorted([f for f in os.listdir(annotations) if f.endswith('.txt')])
# 対応する画像ファイル名を確認 (拡張子はoriginalフォルダから取得)
img_list = sorted([f for f in os.listdir(originals) if os.path.splitext(f)[0] in [os.path.splitext(a)[0] for a in read_files_annotations]])

# 3. データ分割 (コード1のロジック: 2段階分割)
# まず全体を train_test と valid に分割
ann_train_test, ann_valid, img_train_test, img_valid = train_test_split(
    read_files_annotations, img_list, test_size=valid分割割合, random_state=42
)
# 次に train_test を train と test に分割
ann_train, ann_test, img_train, img_test = train_test_split(
    ann_train_test, img_train_test, test_size=test分割割合, random_state=42
)

split_data = {
    'train': (img_train, ann_train),
    'valid': (img_valid, ann_valid),
    'test':  (img_test, ann_test)
}

# 4. 処理の実行
aug_pipeline = get_aug_pipeline()

for split_name, (images, anns) in split_data.items():
    print(f"\nProcessing {split_name}...")
    for img_name, ann_name in tqdm(zip(images, anns), total=len(images)):
        img_src = os.path.join(originals, img_name)
        ann_src = os.path.join(annotations, ann_name)

        # 出力先パス
        img_dst = os.path.join(paths[split_name]['images'], img_name)
        ann_dst = os.path.join(paths[split_name]['labels'], ann_name)

        # --- オリジナルをコピー ---
        shutil.copy(img_src, img_dst)
        shutil.copy(ann_src, ann_dst)

        # --- 拡張処理 (trainのみ適用) ---
        if apply_augmentation and split_name == 'train':
            image = cv2.imread(img_src)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            bboxes, class_labels = load_yolo_labels(ann_src)

            for i in range(num_augmented_per_image):
                try:
                    augmented = aug_pipeline(image=image, bboxes=bboxes, class_labels=class_labels)
                    if len(augmented['bboxes']) > 0:
                        aug_basename = f"{os.path.splitext(img_name)[0]}_aug_{i}"
                        # 保存
                        cv2.imwrite(os.path.join(paths['train']['images'], f"{aug_basename}.jpg"),
                                    cv2.cvtColor(augmented['image'], cv2.COLOR_RGB2BGR))
                        save_yolo_labels(os.path.join(paths['train']['labels'], f"{aug_basename}.txt"),
                                         augmented['bboxes'], class_labels)
                except: continue

# 5. 統計表示と最大公約数 (コード1準拠)
counts = {k: len(os.listdir(v['images'])) for k, v in paths.items()}
print(f"\n--- 分割結果 ---")
for k, v in counts.items():
    print(f"{k}/images: {v} 枚")
    print(f"{k}/labels: {len(os.listdir(paths[k]['labels']))} 枚")

greatest_common_divisor = math.gcd(counts['train'], counts['valid'])
print(f"\n最大公約数は {greatest_common_divisor} です。")

# 6. data.yamlの作成 (コード2準拠の形式)
with open(classes_file, 'r') as f:
    class_names = [line.strip() for line in f.readlines() if line.strip()]

yaml_content = f"""
# 検出する項目の名前
names:
{yaml.dump(class_names, default_flow_style=False)}
# 検出する項目の数
nc: {len(class_names)}

# パス設定 (相対パス)
test: ../test/images
train: ../train/images
val: ../valid/images
"""

with open(f"{output_root}/data.yaml", "w") as f:
    f.write(yaml_content.strip())

print(f"\n✅ {output_root}/data.yaml を作成しました。")

#魚眼補正